# 04 · Portfolio Optimization

Builds four classical portfolios using SLSQP with:
- Single-fund cap (default 40%)
- Category caps (Sectoral ≤ 20%, Thematic ≤ 25%, International ≤ 25%, Commodity ≤ 20%)
- Long-only, fully invested

Covariance uses **Ledoit–Wolf shrinkage** when ≥ 2 assets to avoid singularities.

In [ ]:
# Bootstrap path so the notebook finds the src/ package
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
print("Project root:", ROOT)


In [ ]:
from data_loader import load_all, LoaderConfig
from feature_engineering import compute_fund_features
from optimizer import PortfolioOptimizer, OptimizerConfig, build_return_matrix, annualized_cov, annualized_mu

data = load_all(LoaderConfig(offline_mode=True, lookback_days=1260))
features = compute_fund_features(data["universe"], data["navs"], data["benchmarks"])
codes = features.scheme_code.tolist()
rets = build_return_matrix(data["navs"], codes)
mu = annualized_mu(rets)
cov = annualized_cov(rets)
cats = data["universe"].set_index("scheme_code")["category"]
opt = PortfolioOptimizer(mu, cov, cats, OptimizerConfig())
print(f"{len(codes)} funds · {rets.shape[0]} trading days")

### Solve all four portfolios

In [ ]:
import numpy as np, pandas as pd

portfolios = {
    "Max Sharpe":      opt.max_sharpe(),
    "Min Volatility":  opt.min_volatility(),
    "Max Return":      opt.max_return(),
    "Risk Parity":     opt.risk_parity(),
}
summary = pd.DataFrame({k: opt.summarize(w) for k, w in portfolios.items()}).T
summary

### Efficient frontier

In [ ]:
import numpy as np, matplotlib.pyplot as plt

target_rets = np.linspace(mu.min(), mu.max(), 25)
frontier = []
for r in target_rets:
    w = opt.target_return(float(r))
    frontier.append(opt.summarize(w))
frontier = pd.DataFrame(frontier)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(np.sqrt(np.diag(cov)), mu, c="lightgray", s=40, label="individual funds")
ax.plot(frontier["volatility"], frontier["expected_return"], "-", color="#4B7BEC", lw=2, label="efficient frontier")
for name, w in portfolios.items():
    s = opt.summarize(w); ax.scatter(s["volatility"], s["expected_return"], s=120, label=name)
ax.set_xlabel("Annualized Volatility"); ax.set_ylabel("Expected Return")
ax.set_title("Efficient Frontier"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Allocations for the Max Sharpe portfolio

In [ ]:
w = portfolios["Max Sharpe"]
df = pd.DataFrame({"scheme_code": codes, "weight": w})\
       .merge(data["universe"][["scheme_code","scheme_name","category"]], on="scheme_code")\
       .query("weight > 0.015").sort_values("weight", ascending=False)
df